# Mem0: Universal Memory Layer for AI Agents — Complete Tutorial Notebook

> **Companion notebook to `mem0-tutorial.md`**  
> Covers: Installation → Basic CRUD → Search → Configuration → Advanced Features → Real-World Patterns

## Prerequisites

- Python 3.10+
- An OpenAI API key (or another supported LLM provider)
- `pip install mem0ai`

## How to Use This Notebook

Run cells sequentially. Each section builds on the previous one. Cells marked with `[DEMO]` require an API key and will make real API calls. Cells marked with `[CONCEPT]` are code patterns you can adapt.

---

## 1. Installation & First Memory

Let's install Mem0 and store our first memory.

In [ ]:
# [SETUP] Install mem0ai (run this cell once)
# If you already have it installed, this is a no-op
!pip install mem0ai -q
print("✓ mem0ai installed (or already present)")

In [ ]:
# [SETUP] Import and configure
import os
from mem0 import Memory

# Set your OpenAI API key here (or export OPENAI_API_KEY before starting)
# os.environ["OPENAI_API_KEY"] = "sk-..."  # <-- replace with your key

# Check that API key is available
if not os.environ.get("OPENAI_API_KEY"):
    print("⚠️  OPENAI_API_KEY not set. Set it with: os.environ['OPENAI_API_KEY'] = 'sk-...'")
    print("    Or export it before starting Jupyter: export OPENAI_API_KEY='sk-...'")
else:
    print("✓ OPENAI_API_KEY is set")

In [ ]:
# [DEMO] Initialize Mem0 with defaults and store a first memory
#
# Default configuration:
#   LLM: OpenAI gpt-5-mini (for fact extraction)
#   Embedder: OpenAI text-embedding-3-small (1536 dimensions)
#   Vector store: Qdrant (local, at /tmp/qdrant)
#   History store: SQLite (at ~/.mem0/history.db)
#   Reranker: disabled

memory = Memory()
print("✓ Memory initialized with defaults")
print(f"  History DB: ~/.mem0/history.db")
print(f"  Vector store: /tmp/qdrant")

In [ ]:
# [DEMO] Add our first memory — the LLM will extract key facts from the conversation

messages = [
    {"role": "user", "content": "Hi! I'm Alex. I work as a data engineer at Netflix. I love basketball and sci-fi movies. I'm also a vegetarian."},
    {"role": "assistant", "content": "Great to meet you, Alex! I'll remember all of that."}
]

# add() extracts facts and stores them
result = memory.add(messages, user_id="alex")

print("Extracted memories:")
for item in result:
    print(f"  [{item.get('event', 'ADD')}] {item['memory']}")
    print(f"       id: {item['id']}")

In [ ]:
# [DEMO] Search for those memories

query = "What does Alex do for work and what are his dietary preferences?"
results = memory.search(query, filters={"user_id": "alex"})

print(f"Query: '{query}'")
print(f"Results: {len(results['results'])} found\n")
for mem in results["results"]:
    print(f"  Score: {mem['score']:.3f} | {mem['memory']}")
    print(f"  Categories: {mem.get('categories', [])}")
    print(f"  Created: {mem.get('created_at', 'N/A')}")
    print()

## 2. Adding Memories — Deep Dive

The `add()` method is the core of Mem0. Let's explore all its options.

In [ ]:
# [DEMO] Adding memories with METADATA
# Metadata attaches custom key-value pairs to every extracted memory.
# This enables powerful filtering later.

messages = [
    {"role": "user", "content": "I'm planning a trip to Tokyo in August. I prefer window seats and I'm a vegetarian."},
    {"role": "assistant", "content": "Noted! I'll keep your preferences in mind for trip planning."}
]

result = memory.add(
    messages,
    user_id="alex",
    metadata={
        "category": "travel",
        "topic": "preferences",
        "source": "trip_planning_session"
    }
)

print("Memories stored with metadata:")
for item in result:
    print(f"  [{item.get('event', 'ADD')}] {item['memory']}")

In [ ]:
# [DEMO] Raw insert (infer=False) — stores messages VERBATIM without LLM extraction
# Use this when you want to preserve exact message text, not extracted facts.

messages = [
    {"role": "user", "content": "The exact transcript: 'I arrived at 3:45 PM and the server temperature was 42.7°C.'"}
]

result = memory.add(messages, user_id="alex", infer=False)
print("Raw insert result:", result)

In [ ]:
# [DEMO] Adding with agent_id and run_id for multi-agent scenarios
# Different IDs help isolate memories by agent, session, and app.

messages = [
    {"role": "user", "content": "Remind me to check the load balancer at 6 PM."},
    {"role": "assistant", "content": "Will do! I'll set a reminder."}
]

result = memory.add(
    messages,
    user_id="alex",
    agent_id="devops-bot",      # which agent handled this
    run_id="session-2026-06-22"  # which session/run
)

print(f"Stored {len(result)} facts under user=alex, agent=devops-bot, run=session-2026-06-22")

## 3. Searching Memories — Deep Dive

Mem0's hybrid search fuses semantic, keyword, and entity matching.

In [ ]:
# [DEMO] Basic search with top_k and threshold

query = "What are Alex's food preferences and travel plans?"

results = memory.search(
    query,
    filters={"user_id": "alex"},
    top_k=5,        # return up to 5 results
    threshold=0.1   # only results with score >= 0.1
)

print(f"Query: '{query}'")
print(f"Found {len(results['results'])} results:\n")
for i, mem in enumerate(results["results"], 1):
    print(f"{i}. [score={mem['score']:.3f}] {mem['memory']}")

In [ ]:
# [DEMO] Temporal search — bias toward recent or time-anchored memories
# Use reference_date when questions like "what did I say last week?" need temporal context.

# Search anchored to a specific date
results = memory.search(
    "What is Alex's job?",
    filters={"user_id": "alex"},
    reference_date="2026-06-22"  # ISO date format
)

print("Temporally-anchored search results:")
for mem in results["results"]:
    print(f"  [{mem['score']:.3f}] {mem['memory']}")

# Unix epoch also works
import time
results_epoch = memory.search(
    "What is Alex's job?",
    filters={"user_id": "alex"},
    reference_date=int(time.time())
)
print(f"\n(Unix epoch search returned {len(results_epoch['results'])} results)")

## 4. CRUD — Get, Update, Delete

Full lifecycle management of stored memories.

In [ ]:
# [DEMO] GET — Retrieve all memories for a user (paginated)

memories = memory.get_all(
    filters={"user_id": "alex"},
    page=1,
    page_size=50
)

print(f"Total memories for Alex: {memories['count']}")
print(f"Showing page {1} (first {len(memories['results'])} items):\n")
for mem in memories["results"]:
    cats = mem.get('categories', [])
    print(f"  [{mem['id'][:8]}...] {mem['memory']}")
    if cats:
        print(f"          Categories: {cats}")
    print(f"          Created: {mem.get('created_at', 'N/A')}")
    print()

In [ ]:
# [DEMO] GET — Retrieve a specific memory by ID

# First, get a memory ID from search or get_all
memories = memory.get_all(filters={"user_id": "alex"}, page=1, page_size=1)
if memories["results"]:
    sample_id = memories["results"][0]["id"]
    
    # Now get that specific memory
    single_mem = memory.get(memory_id=sample_id)
    print(f"Single memory by ID ({sample_id[:12]}...):")
    print(f"  Memory: {single_mem['memory']}")
    print(f"  User: {single_mem.get('user_id')}")
    print(f"  Created: {single_mem.get('created_at')}")
else:
    print("No memories found to demonstrate get()")
    print("Run the add() cells above first.")

In [ ]:
# [DEMO] UPDATE — Modify a memory's content
# First add something we'll update

add_result = memory.add(
    [{"role": "user", "content": "My favorite coffee is latte."}],
    user_id="alex"
)

if add_result:
    mem_id = add_result[0]["id"]
    old_text = add_result[0]["memory"]
    print(f"Added: {old_text} (id: {mem_id[:12]}...)")
    
    # Now update it — Alex changed his preference
    update_result = memory.update(
        memory_id=mem_id,
        data="Alex now prefers cold brew coffee instead of latte"
    )
    print(f"Updated: {update_result}")
else:
    print("No fact extracted from the message")

In [ ]:
# [DEMO] DELETE — Remove a specific memory

# Add a temporary memory
temp = memory.add(
    [{"role": "user", "content": "Temporary note: remember to buy milk."}],
    user_id="alex"
)

if temp:
    temp_id = temp[0]["id"]
    print(f"Added temp memory: {temp[0]['memory']}")
    
    # Delete it
    delete_result = memory.delete(memory_id=temp_id)
    print(f"Deleted: {delete_result}")
    
    # Verify it's gone (will raise ValueError)
    try:
        memory.get(memory_id=temp_id)
        print("⚠️  Memory still exists!")
    except ValueError:
        print("✓ Memory successfully deleted (get raises ValueError)")
else:
    print("No temp memory created")

In [ ]:
# [DEMO] DELETE_ALL — Bulk delete by filter
# WARNING: This is DESTRUCTIVE. Use with care.

# Delete all memories for a specific run_id (session cleanup)
# memory.delete_all(user_id="alex", run_id="old-session-123")

# Delete all for a user
# memory.delete_all(user_id="alex")  # <-- uncomment with caution!

print("⚠️  delete_all examples are commented out for safety.")
print("    Uncomment them when you want to use them.")

In [ ]:
# [DEMO] HISTORY — View change log for a memory

# Let's find a memory that we updated earlier
search_result = memory.search("cold brew", filters={"user_id": "alex"}, top_k=1)
if search_result["results"]:
    mem_id = search_result["results"][0]["id"]
    print(f"History for memory: {search_result['results'][0]['memory']}\n")
    
    history = memory.history(memory_id=mem_id)
    for event in history:
        print(f"  [{event.get('event', 'UNKNOWN')}] at {event.get('created_at', 'N/A')}")
        if 'old_memory' in event:
            print(f"    Old: {event['old_memory']}")
        if 'new_memory' in event:
            print(f"    New: {event['new_memory']}")
else:
    print("No 'cold brew' memory found. Run the update cell above first.")

## 5. Configuration — Custom LLMs, Embedders & Vector Stores

Mem0 OSS is fully configurable. Swap any component.

In [ ]:
# [CONCEPT] Configuration via Python dict
#
# Available LLM providers: openai, anthropic, ollama, groq, deepseek,
#                         google, bedrock, together, azure_openai, litellm,
#                         xai, lmstudio, langchain, and more
#
# Available embedders: openai, ollama, huggingface, google, vertexai,
#                      together, bedrock, azure_openai, lmstudio, langchain
#
# Available vector stores: qdrant, chroma, weaviate, pinecone, milvus,
#                          redis, pgvector, neo4j, faiss

config = {
    # ── Extraction LLM ──────────────────────────
    # Keep temperature LOW (≤0.2) for deterministic fact extraction
    "llm": {
        "provider": "openai",
        "config": {
            "model": "gpt-4.1-mini",   # or gpt-5-mini, gpt-4o-mini
            "temperature": 0.1,
            "max_tokens": 2000
        }
    },
    
    # ── Embedding Model ─────────────────────────
    "embedder": {
        "provider": "openai",
        "config": {
            "model": "text-embedding-3-small",  # 1536 dimensions
            # "embedding_dims": 1536  # optional, auto-detected
        }
    },
    
    # ── Vector Store ────────────────────────────
    "vector_store": {
        "provider": "qdrant",
        "config": {
            "host": "localhost",
            "port": 6333,
            "collection_name": "my_app_memories"  # isolate tenants
        }
    },
    
    # ── History Store ───────────────────────────
    "history_db_path": "~/.mem0/my_app_history.db"
}

# Create a configured memory instance
# memory_custom = Memory.from_config(config)

print("✓ Config dict defined (not applied — uncomment Memory.from_config to use)")
print(f"  LLM: {config['llm']['provider']} / {config['llm']['config']['model']}")
print(f"  Embedder: {config['embedder']['provider']} / {config['embedder']['config']['model']}")
print(f"  Vector store: {config['vector_store']['provider']}")

In [ ]:
# [CONCEPT] Alternative provider configurations
# Uncomment the one you want to use.

# ── Anthropic (Claude) ──────────────────────────
# config_anthropic = {
#     "llm": {
#         "provider": "anthropic",
#         "config": {
#             "model": "claude-sonnet-4-6",
#             "temperature": 0.1
#         }
#     }
# }

# ── Ollama (local models) ───────────────────────
# config_ollama = {
#     "llm": {
#         "provider": "ollama",
#         "config": {
#             "model": "qwen3:32b",
#             "ollama_base_url": "http://localhost:11434"
#         }
#     },
#     "embedder": {
#         "provider": "ollama",
#         "config": {
#             "model": "nomic-embed-text",
#             "embedding_dims": 768
#         }
#     }
# }

# ── DeepSeek ────────────────────────────────────
# config_deepseek = {
#     "llm": {
#         "provider": "deepseek",
#         "config": {
#             "model": "deepseek-chat",
#             "api_key": os.environ.get("DEEPSEEK_API_KEY")
#         }
#     }
# }

# ── Google Gemini ───────────────────────────────
# config_gemini = {
#     "llm": {
#         "provider": "google",
#         "config": {
#             "model": "gemini-2.0-flash",
#             "api_key": os.environ.get("GOOGLE_API_KEY")
#         }
#     }
# }

# ── Groq (fast inference) ───────────────────────
# config_groq = {
#     "llm": {
#         "provider": "groq",
#         "config": {
#             "model": "llama-3.1-70b-versatile",
#             "api_key": os.environ.get("GROQ_API_KEY")
#         }
#     }
# }

print("Alternative configs defined. Uncomment to activate.")
print("Providers: Anthropic, Ollama, DeepSeek, Google Gemini, Groq")

In [ ]:
# [CONCEPT] YAML Configuration
# Mem0 also supports loading config from a YAML file.
# Create a config.yaml with your settings, then:
#
#   memory = Memory.from_config_file("config.yaml")
#
# Example config.yaml content:

example_yaml = '''
vector_store:
  provider: qdrant
  config:
    host: localhost
    port: 6333

llm:
  provider: openai
  config:
    model: gpt-4.1-mini
    temperature: 0.1

embedder:
  provider: openai
  config:
    model: text-embedding-3-small
'''

print("Example config.yaml:")
print(example_yaml)
print("Load with: memory = Memory.from_config_file('config.yaml')")

## 6. Advanced Metadata Filtering

Mem0's filter DSL supports logical operators, comparisons, substring matching, and wildcards.

In [ ]:
# [DEMO] Let's add some categorized memories to demonstrate filtering

# Work-related
memory.add(
    [{"role": "user", "content": "I'm leading the data pipeline migration project. Priority: HIGH."}],
    user_id="alex", metadata={"category": "work", "priority": "high"}
)

# Health-related
memory.add(
    [{"role": "user", "content": "My allergies have been acting up. I need to schedule a doctor appointment."}],
    user_id="alex", metadata={"category": "health", "priority": "medium"}
)

# Hobby-related
memory.add(
    [{"role": "user", "content": "I just finished reading Dune. Amazing sci-fi novel!"}],
    user_id="alex", metadata={"category": "hobby", "priority": "low"}
)

print("✓ Added 3 categorized memories (work, health, hobby)")

In [ ]:
# [DEMO] Filtering: IN / NIN (list-based matching)

# Search across specific categories
results = memory.search(
    "What is Alex working on?",
    filters={
        "user_id": "alex",
        "category": {"in": ["work", "health"]}  # only work or health
    }
)

print(f"Results (work OR health only): {len(results['results'])}")
for mem in results["results"]:
    print(f"  [{mem['score']:.3f}] {mem['memory']}")

In [ ]:
# [DEMO] Filtering: Logical AND / OR combinations

# AND: all conditions must match
results_and = memory.search(
    "What should I prioritize?",
    filters={
        "AND": [
            {"user_id": "alex"},
            {"category": {"in": ["work"]}},
            {"priority": "high"}
        ]
    }
)
print(f"AND filter (work + high priority): {len(results_and['results'])} results")
for mem in results_and["results"]:
    print(f"  {mem['memory']}")

# OR: any condition matches
results_or = memory.search(
    "What should I know?",
    filters={
        "user_id": "alex",
        "OR": [
            {"category": "health"},
            {"category": "work"}
        ]
    }
)
print(f"\nOR filter (health OR work): {len(results_or['results'])} results")
for mem in results_or["results"]:
    print(f"  {mem['memory']}")

In [ ]:
# [DEMO] Filtering: NOT (exclusion)

results_not = memory.search(
    "What is Alex interested in?",
    filters={
        "user_id": "alex",
        "NOT": [
            {"category": "work"}  # exclude work-related
        ]
    }
)

print(f"NOT work filter: {len(results_not['results'])} results")
for mem in results_not["results"]:
    print(f"  [{mem['score']:.3f}] {mem['memory']}")

In [ ]:
# [CONCEPT] Complex nested logic — AND(OR(...), NOT(...))

results_nested = memory.search(
    "Tell me about Alex",
    filters={
        "AND": [
            {"user_id": "alex"},
            {
                "OR": [
                    {"category": "work"},
                    {"category": "health"}
                ]
            },
            {
                "NOT": [
                    {"priority": "low"}
                ]
            }
        ]
    }
)

print("Nested filter: (work OR health) AND NOT low priority")
print(f"Results: {len(results_nested['results'])}")
for mem in results_nested["results"]:
    print(f"  {mem['memory']}")

In [ ]:
# [DEMO] Comparison operators (gt, gte, lt, lte, eq, ne)
# These work on numeric metadata fields

# First, let's add some memories with numeric scores
memory.add(
    [{"role": "user", "content": "Feature X has a confidence score of 0.95."}],
    user_id="alex", metadata={"type": "analysis", "confidence": 0.95}
)
memory.add(
    [{"role": "user", "content": "Feature Y has a confidence score of 0.45."}],
    user_id="alex", metadata={"type": "analysis", "confidence": 0.45}
)

# Now filter: only high-confidence memories
results = memory.search(
    "feature confidence scores",
    filters={
        "user_id": "alex",
        "confidence": {"gte": 0.8}  # confidence >= 0.8
    }
)

print("High-confidence features (>= 0.8):")
for mem in results["results"]:
    print(f"  [{mem['score']:.3f}] {mem['memory']}")

In [ ]:
# [DEMO] String operators: contains / icontains

memory.add(
    [{"role": "user", "content": "Meeting: Q2 roadmap review — discuss timeline and milestones."}],
    user_id="alex", metadata={"title": "Q2 Roadmap Review Meeting", "type": "meeting_notes"}
)

results = memory.search(
    "meeting about roadmap",
    filters={
        "user_id": "alex",
        "title": {"icontains": "roadmap"}  # case-insensitive substring match
    }
)

print("Meetings with 'roadmap' in title:")
for mem in results["results"]:
    print(f"  {mem['memory']}")

In [ ]:
# [DEMO] Wildcard (*) — match any value (field exists)

# All memories that have ANY category
results = memory.search(
    "anything",
    filters={"user_id": "alex", "category": "*"}
)

print(f"Memories with any category: {len(results['results'])}")
print("(The wildcard '*' matches all memories that have the 'category' field set)")

## 7. Reranker-Enhanced Search

Rerankers add a second scoring pass for better precision.

In [ ]:
# [CONCEPT] Reranker configuration
#
# Reranker options:
#   - Cohere (API, paid): rerank-english-v3.0
#   - Sentence Transformer (local, free): cross-encoder/ms-marco-MiniLM-L-6-v2
#   - HuggingFace (local, free): various cross-encoder models
#   - LLM Reranker (API, paid): uses gpt-4o-mini to re-rank
#   - Zero Entropy (API, paid)

# ── Cohere Reranker ────────────────────────────
config_cohere_rerank = {
    "reranker": {
        "provider": "cohere",
        "config": {
            "model": "rerank-english-v3.0",
            "api_key": os.environ.get("COHERE_API_KEY", "your-cohere-key")
        }
    }
}

# ── Local Sentence Transformer (FREE, no API key needed) ──
config_local_rerank = {
    "reranker": {
        "provider": "sentence_transformer",
        "config": {
            "model": "cross-encoder/ms-marco-MiniLM-L-6-v2",
            "device": "cpu",          # or "cuda" for GPU
            "max_length": 512
        }
    }
}

# ── LLM-based Reranker ─────────────────────────
config_llm_rerank = {
    "reranker": {
        "provider": "llm_reranker",
        "config": {
            "provider": "openai",
            "model": "gpt-4o-mini",
            "top_k": 5
        }
    }
}

print("Reranker configs defined:")
print("  1. Cohere (API, paid)")
print("  2. Sentence Transformer (local, free)")
print("  3. LLM Reranker (API, paid)")
print("\nUsage: memory.search(..., rerank=True)")

In [ ]:
# [CONCEPT] Smart reranking — only use it for complex queries

def smart_search(memory, query, user_id):
    """
    Use reranker only for complex multi-word queries.
    Simple single-word queries skip reranking to save latency/cost.
    """
    use_rerank = len(query.split()) > 3  # rerank only if > 3 words
    
    if use_rerank:
        print(f"  (using reranker — {len(query.split())} words)")
    else:
        print(f"  (skipping reranker — {len(query.split())} words)")
    
    return memory.search(
        query,
        filters={"user_id": user_id},
        rerank=use_rerank
    )

# Example usage
# results = smart_search(memory, "preferences", "alex")  # no rerank
# results = smart_search(memory, "What are Alex's work projects and dietary preferences?", "alex")  # rerank

print("✓ smart_search function defined")
print("  Short queries → no rerank (faster, cheaper)")
print("  Long queries → rerank (better precision)")

In [ ]:
# [CONCEPT] Reranker fallback pattern — never lose a search to reranker failure

def search_with_fallback(memory, query, filters, use_rerank=True):
    """
    Try search with reranker. If it fails, fall back to vector-only search.
    This ensures users always get results, even if the reranker is down.
    """
    if not use_rerank:
        return memory.search(query, filters=filters, rerank=False)
    
    try:
        results = memory.search(query, filters=filters, rerank=True)
        print("✓ Reranked search succeeded")
        return results
    except Exception as e:
        print(f"⚠️  Reranker failed ({e}), falling back to vector-only search")
        return memory.search(query, filters=filters, rerank=False)

print("✓ search_with_fallback function defined")
print("  Try reranker → if fail → fallback to vector-only")

## 8. Async Memory — Non-Blocking Operations

For FastAPI, background workers, and async applications.

In [ ]:
# [CONCEPT] AsyncMemory basics

import asyncio
from mem0 import AsyncMemory

# Initialize async memory (same config patterns as sync Memory)
# async_memory = AsyncMemory()  # defaults
# async_memory = AsyncMemory(config=custom_config)  # custom config

print("AsyncMemory API (all methods are awaitable):")
print("  await memory.add(messages, user_id=...)")
print("  await memory.search(query, filters={...})")
print("  await memory.get_all(filters={...})")
print("  await memory.get(memory_id=...)")
print("  await memory.update(memory_id=..., data=...)")
print("  await memory.delete(memory_id=...)")
print("  await memory.delete_all(user_id=...)")
print("  await memory.history(memory_id=...)")

In [ ]:
# [DEMO] AsyncMemory — concurrent batch operations
# This demonstrates running multiple add() calls in parallel.
# Skip if OPENAI_API_KEY is not set (will fail on the actual API calls).

async def demo_async_batch():
    """Add memories for 5 users concurrently using asyncio.gather."""
    
    # Check for API key
    if not os.environ.get("OPENAI_API_KEY"):
        print("⚠️  Skipping: OPENAI_API_KEY not set")
        print("    Set it and re-run to see async in action.")
        return
    
    memory = AsyncMemory()
    
    # Prepare 5 concurrent tasks
    users_and_preferences = [
        ("user_1", "I love hiking and photography."),
        ("user_2", "I'm a software engineer who loves cooking."),
        ("user_3", "I play tennis every weekend."),
        ("user_4", "I'm learning Japanese and love anime."),
        ("user_5", "I'm a freelance designer who works remotely."),
    ]
    
    tasks = [
        memory.add(
            messages=[{"role": "user", "content": preference}],
            user_id=user_id
        )
        for user_id, preference in users_and_preferences
    ]
    
    print(f"Launching {len(tasks)} concurrent add() calls...")
    results = await asyncio.gather(*tasks, return_exceptions=True)
    
    for i, result in enumerate(results):
        user_id = users_and_preferences[i][0]
        if isinstance(result, Exception):
            print(f"  ✗ {user_id}: FAILED — {result}")
        else:
            facts_count = len(result) if isinstance(result, list) else 0
            print(f"  ✓ {user_id}: {facts_count} facts stored")

# Run the async demo
# asyncio.run(demo_async_batch())  # uncomment to run

print("✓ demo_async_batch() coroutine defined")
print("  Uncomment asyncio.run(demo_async_batch()) to execute")
print("  Requires: OPENAI_API_KEY set")

In [ ]:
# [CONCEPT] Async resilience — timeout + retry pattern

async def with_timeout_and_retry(operation, max_retries=3, timeout=10.0):
    """
    Execute an async operation with timeout and exponential backoff.
    
    Args:
        operation: async callable (e.g., lambda: memory.search(...))
        max_retries: maximum retry attempts
        timeout: seconds before timeout per attempt
    
    Returns:
        Result of the operation
    
    Raises:
        Exception if all retries exhausted
    """
    for attempt in range(max_retries):
        try:
            return await asyncio.wait_for(operation(), timeout=timeout)
        except asyncio.TimeoutError:
            print(f"  Timeout on attempt {attempt + 1}/{max_retries}")
        except Exception as exc:
            print(f"  Error on attempt {attempt + 1}/{max_retries}: {exc}")
        
        if attempt < max_retries - 1:
            delay = 2 ** attempt  # exponential backoff: 1s, 2s, 4s
            print(f"  Retrying in {delay}s...")
            await asyncio.sleep(delay)
    
    raise Exception(f"Operation failed after {max_retries} attempts")

# Usage:
# result = await with_timeout_and_retry(
#     lambda: async_memory.search("query", filters={"user_id": "alex"}),
#     max_retries=3,
#     timeout=10.0
# )

print("✓ with_timeout_and_retry() defined")
print("  Exponential backoff: 1s → 2s → 4s")
print("  Timeout per attempt: 10s")

## 9. Graph Memory

Connect entities (people, places, organizations) with relationships.

In [ ]:
# [CONCEPT] Graph Memory setup with Neo4j
# Requires: pip install "mem0ai[graph]"
# Requires: A running Neo4j instance (local or Aura cloud)

graph_config = {
    "graph_store": {
        "provider": "neo4j",
        "config": {
            "url": os.environ.get("NEO4J_URL", "bolt://localhost:7687"),
            "username": os.environ.get("NEO4J_USERNAME", "neo4j"),
            "password": os.environ.get("NEO4J_PASSWORD", "password"),
            "database": "neo4j",
        },
        # Optional custom prompt to guide entity extraction:
        # "custom_prompt": "Only capture people, organizations, and project links."
    }
}

print("Graph Memory providers:")
print("  - Neo4j (graph-native, cloud or self-hosted)")
print("  - Memgraph (in-memory, low-latency)")
print("  - Neptune (AWS managed)")
print("  - Kuzu (embedded, no server needed)")
print("  - Apache AGE (PostgreSQL extension)")
print()
print("Key controls:")
print("  memory.add(messages, ..., enable_graph=False)  # skip graph")
print("  memory.search(..., enable_graph=False)         # skip graph lookup")
print("  config['graph_store']['custom_prompt'] = '...'  # guide extraction")

In [ ]:
# [CONCEPT] Graph Memory example flow (requires Neo4j)
#
# memory = Memory.from_config(graph_config)
#
# conversation = [
#     {"role": "user", "content": "Alice met Bob at GraphConf 2025 in San Francisco."},
#     {"role": "assistant", "content": "Great! Logging that connection."},
# ]
# memory.add(conversation, user_id="demo-user")
#
# results = memory.search(
#     "Who did Alice meet at GraphConf?",
#     filters={"user_id": "demo-user"}
# )
#
# for hit in results["results"]:
#     print(hit["memory"])
#     if "relations" in hit:
#         print("  Relations:", hit["relations"])

print("Graph Memory example defined.")
print("Requires: pip install 'mem0ai[graph]' + running Neo4j instance.")
print()
print("Verify in Neo4j Browser:")
print("  MATCH (p:Person)-[r]->(q:Person) RETURN p, r, q LIMIT 10;")
print("  MATCH (n) RETURN n LIMIT 25;")

## 10. Multimodal Support — Images

Mem0 can extract memories from images (receipts, screenshots, photos).

In [ ]:
# [CONCEPT] Adding memories from image URLs

# Images in messages use the same format as OpenAI's vision API
messages_with_image_url = [
    {"role": "user", "content": "Here's the menu from the restaurant I visited."},
    {
        "role": "user",
        "content": {
            "type": "image_url",
            "image_url": {"url": "https://example.com/restaurant-menu.jpg"}
        }
    }
]

# memory.add(messages_with_image_url, user_id="alex")

print("Image URL format (OpenAI Vision API compatible):")
print('  {"type": "image_url", "image_url": {"url": "..."}}')
print()
print("Supported formats: JPEG, PNG, WebP, GIF")
print("Max file size: 20 MB (recommended: < 5 MB)")

In [ ]:
# [CONCEPT] Adding local images with base64 encoding

import base64

def encode_image_to_base64(image_path):
    """
    Read a local image file and encode it as base64 for Mem0.
    
    Args:
        image_path: Path to the image file (JPEG, PNG, etc.)
    
    Returns:
        Base64-encoded string with data URI prefix
    """
    # Determine MIME type from extension
    ext = image_path.lower().split('.')[-1]
    mime_map = {
        'jpg': 'image/jpeg',
        'jpeg': 'image/jpeg',
        'png': 'image/png',
        'webp': 'image/webp',
        'gif': 'image/gif'
    }
    mime_type = mime_map.get(ext, 'image/jpeg')
    
    with open(image_path, "rb") as f:
        encoded = base64.b64encode(f.read()).decode("utf-8")
    
    return f"data:{mime_type};base64,{encoded}"

# Usage:
# base64_img = encode_image_to_base64("receipt.jpg")
# messages = [{
#     "role": "user",
#     "content": [
#         {"type": "text", "text": "What's in this receipt?"},
#         {"type": "image_url", "image_url": {"url": base64_img}}
#     ]
# }]
# memory.add(messages, user_id="alex")

print("✓ encode_image_to_base64() defined")
print("  Supports: jpg, jpeg, png, webp, gif")
print("  Usage: encode local images before passing to memory.add()")

In [ ]:
# [CONCEPT] Multimodal error handling

def safe_multimodal_add(memory, messages, user_id):
    """
    Add multimodal messages with proper error handling.
    Catches common image-related errors.
    """
    # Client-side size check (if you have the image path)
    # import os
    # if os.path.getsize(image_path) > 20 * 1024 * 1024:  # 20 MB
    #     raise ValueError("Image exceeds 20 MB limit")
    
    try:
        from mem0.exceptions import InvalidImageError, FileSizeError
        result = memory.add(messages, user_id=user_id)
        print(f"✓ Added {len(result)} facts from multimodal message")
        return result
    except FileSizeError:
        print("✗ Image too large (max 20 MB). Resize and try again.")
        return []
    except InvalidImageError:
        print("✗ Invalid image format or corrupted file.")
        return []
    except ImportError:
        # mem0.exceptions may not be available in older versions
        result = memory.add(messages, user_id=user_id)
        return result
    except Exception as e:
        print(f"✗ Unexpected error: {e}")
        return []

print("✓ safe_multimodal_add() defined")
print("  Handles: FileSizeError, InvalidImageError, general exceptions")

In [ ]:
# [BEST PRACTICE] Multimodal tips

print("Multimodal Best Practices:")
print()
print("1. ALWAYS include a text message alongside images")
print("   ✓ messages = [{'role': 'user', 'content': 'Here is my receipt'},")
print("                {'role': 'user', 'content': {...image...}}]")
print()
print("2. Keep images under 5 MB for faster processing")
print("   Compress/resize before sending.")
print()
print("3. Split bulk uploads into separate add() calls")
print("   One failure won't affect other images.")
print()
print("4. Validate file size on the client side before encoding")
print("   Saves bandwidth and avoids server-side rejection.")
print()
print("5. Use clear, well-lit images for better extraction accuracy")

## 11. Custom Instructions — Controlling Fact Extraction

Guide the LLM to extract only what matters for your domain.

In [ ]:
# [CONCEPT] Custom instructions — domain-specific extraction

# This prompt tells the extraction LLM to ONLY capture
# order-related facts, ignoring casual conversation.

order_extraction_prompt = """
Please only extract entities containing customer support information,
order details, and shipping information.

Here are few-shot examples:

Input: Hi, how are you?
Output: {"facts": []}

Input: My order #12345 hasn't arrived yet.
Output: {"facts": ["Order #12345 has not arrived"]}

Input: I need to change my shipping address to 123 Main St, Austin TX 78701.
Output: {"facts": ["Shipping address changed to 123 Main St, Austin TX 78701"]}

Input: I love hiking on weekends.
Output: {"facts": []}

Return ONLY the JSON object with a 'facts' key.
Do not extract facts about hobbies, personal interests, or casual conversation.
"""

# Configure memory with custom instructions
config_with_instructions = {
    "llm": {
        "provider": "openai",
        "config": {
            "model": "gpt-4.1-mini",
            "temperature": 0.1  # low temp for deterministic extraction
        }
    },
    "custom_instructions": order_extraction_prompt
}

# memory = Memory.from_config(config_with_instructions)

print("✓ Custom instructions defined for order/shipping extraction")
print()
print("What this does:")
print("  ✓ Captures: order numbers, shipping addresses, issues")
print("  ✗ Ignores: casual chat, hobbies, personal interests")

In [ ]:
# [CONCEPT] Custom instructions — healthcare domain example

healthcare_extraction_prompt = """
Extract ONLY medical and health-related facts from patient conversations.
Focus on: symptoms, medications, allergies, diagnoses, appointment dates,
and doctor recommendations.

Few-shot examples:

Input: I've had a headache for three days and I'm sneezing a lot.
Output: {"facts": ["Patient reports headache for 3 days", "Patient reports frequent sneezing"]}

Input: The doctor prescribed amoxicillin 500mg twice daily for 7 days.
Output: {"facts": ["Prescribed amoxicillin 500mg, twice daily, 7-day course"]}

Input: I'm allergic to penicillin.
Output: {"facts": ["Allergy: penicillin"]}

Input: The weather is nice today.
Output: {"facts": []}

Return ONLY {"facts": [...]} JSON. No other keys.
"""

print("✓ Healthcare-specific custom instructions defined")
print()
print("Design pattern for custom instructions:")
print("  1. State allowed fact types explicitly")
print("  2. Include few-shot examples (positive AND negative)")
print("  3. Show empty [] for irrelevant messages")
print("  4. Require strict JSON: {\"facts\": [...]} only")

In [ ]:
# [CONCEPT] Per-call override of custom instructions
# You can override the global custom_instructions for a single add() call.

per_call_instructions = """
For this call only, extract facts about food preferences and allergies.
Ignore everything else.

Input: I love pizza but I'm allergic to dairy.
Output: {"facts": ["Likes pizza", "Allergic to dairy"]}

Input: My meeting is at 3 PM.
Output: {"facts": []}
"""

# Usage (if memory was configured with global instructions):
# result = memory.add(
#     messages,
#     user_id="alex",
#     custom_instructions=per_call_instructions  # overrides global
# )

print("✓ Per-call custom_instructions override pattern defined")
print("  Use when a specific conversation needs different extraction rules.")
print("  The global instructions still apply to other calls.")

## 12. Memory Client — Platform (Managed API)

Use Mem0's managed cloud for zero-ops production deployments.

In [ ]:
# [CONCEPT] MemoryClient — Platform setup

from mem0 import MemoryClient

# Get your API key from: https://app.mem0.ai/dashboard/api-keys
# client = MemoryClient(api_key="your-mem0-api-key")

# Or via environment variable:
# export MEM0_API_KEY="your...ent = MemoryClient()  # auto-reads MEM0_API_KEY

print("MemoryClient (Platform) setup:")
print("  1. Sign up at https://app.mem0.ai")
print("  2. Get API key from Dashboard → API Keys")
print("  3. client = MemoryClient(api_key='m0-...')")
print()
print("Platform advantages:")
print("  - Zero-ops (no vector store to manage)")
print("  - Visual dashboard at app.mem0.ai")
print("  - Built-in rerankers (just set rerank=True)")
print("  - Webhooks for real-time memory events")
print("  - Batch operations (up to 1000 at once)")
print("  - Organizations & projects for multi-tenancy")

In [ ]:
# [CONCEPT] Platform — Async processing with event tracking
# Platform add() calls are asynchronous — they return an event_id immediately.

# client = MemoryClient(api_key="...")
# result = client.add(messages, user_id="alice")
# # result = {"message": "...", "status": "PENDING", "event_id": "evt-abc123"}
#
# # Poll until complete
# event = client.get_event(result["event_id"])
# # event = {"status": "SUCCEEDED", "memories": [...]}

print("Platform async flow:")
print("  1. client.add() → returns event_id immediately")
print("  2. client.get_event(event_id) → poll for completion")
print("  3. Status: PENDING → PROCESSING → SUCCEEDED | FAILED")
print()
print("Platform-only features:")
print("  - client.batch_update([{memory_id, text}, ...])  # up to 1000")
print("  - client.batch_delete([{memory_id}, ...])        # up to 1000")

## 13. CLI Tool — Quick Operations from Terminal

`mem0` CLI for quick operations without writing code.

In [ ]:
# [CONCEPT] CLI quick reference

print("mem0 CLI Quick Reference:")
print()
print("# Install")
print("  pip install mem0-cli")
print("  npm install -g @mem0/cli")
print()
print("# Agent signup (no email needed!)")
print("  mem0 init --agent --agent-caller my-agent-name")
print()
print("# CRUD")
print("  mem0 add 'I love pizza' --user-id alice")
print("  mem0 search 'food preferences' --user-id alice")
print("  mem0 get --memory-id mem_abc123")
print("  mem0 update --memory-id mem_abc123 --text 'New fact'")
print("  mem0 delete --memory-id mem_abc123")
print()
print("# Advanced")
print("  mem0 get-all --user-id alice --page 1 --page-size 50")
print("  mem0 search 'query' --rerank --top-k 5 --threshold 0.2")
print("  mem0 add 'text' --metadata '{\"source\":\"cli\"}' --infer=false")

## 14. Real-World Patterns & Recipes

Copy-paste patterns for common use cases.

In [ ]:
# [RECIPE] Personalized AI Assistant with Memory

from openai import OpenAI

class PersonalizedAssistant:
    """
    AI assistant that remembers user preferences across sessions.
    
    Flow:
      1. Search Mem0 for relevant past memories
      2. Inject memories into the LLM's system prompt
      3. Generate personalized response
      4. Store new memories from this exchange
    """
    
    def __init__(self, openai_client=None):
        self.memory = Memory()
        self.llm = openai_client or OpenAI()
    
    def chat(self, message: str, user_id: str) -> str:
        # Step 1: Retrieve relevant memories
        relevant = self.memory.search(
            message,
            filters={"user_id": user_id},
            top_k=5
        )
        memories_str = "\n".join(
            f"- {m['memory']}" for m in relevant["results"]
        )
        
        # Step 2: Build system prompt with memories
        system_prompt = f"""You are a helpful, personalized AI assistant.

User Memories (from past conversations):
{memories_str if memories_str else 'No prior memories.'}

Use these memories to personalize your responses.
If memories are irrelevant to the current question, ignore them."""
        
        # Step 3: Generate response
        response = self.llm.chat.completions.create(
            model="gpt-4.1-mini",
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": message}
            ]
        )
        reply = response.choices[0].message.content
        
        # Step 4: Store this exchange as new memories
        self.memory.add([
            {"role": "user", "content": message},
            {"role": "assistant", "content": reply}
        ], user_id=user_id)
        
        return reply

# Usage:
# assistant = PersonalizedAssistant()
# reply = assistant.chat("What's a good restaurant near me?", user_id="alex")

print("✓ PersonalizedAssistant class defined")
print("  Flow: Search memories → Inject into prompt → Respond → Store new memories")

In [ ]:
# [RECIPE] Multi-User Customer Support Bot

class SupportBot:
    """
    Customer support bot with per-user memory.
    
    Features:
      - Searches user's entire support history
      - Uses metadata for ticket tracking
      - Aggregates customer context on demand
    """
    
    def __init__(self):
        self.memory = Memory()
        self.llm = OpenAI()
    
    def handle_ticket(self, message: str, user_id: str, ticket_id: str) -> str:
        # Search user's support history
        user_history = self.memory.search(
            message,
            filters={
                "AND": [
                    {"user_id": user_id},
                    {"category": {"in": ["support", "issue", "preference"]}}
                ]
            },
            top_k=5
        )
        
        # Build context-aware prompt
        context = "\n".join(
            f"- {m['memory']}" for m in user_history["results"]
        )
        
        response = self.llm.chat.completions.create(
            model="gpt-4.1-mini",
            messages=[
                {"role": "system", "content": f"""You are a support agent.
User history:
{context if context else 'First-time customer.'}

Current ticket: {ticket_id}
Be helpful and reference relevant past issues."""},
                {"role": "user", "content": message}
            ]
        )
        reply = response.choices[0].message.content
        
        # Store with ticket metadata
        self.memory.add([
            {"role": "user", "content": message},
            {"role": "assistant", "content": reply}
        ], user_id=user_id, metadata={
            "ticket_id": ticket_id,
            "category": "support"
        })
        
        return reply
    
    def get_customer_context(self, user_id: str) -> dict:
        """Aggregate all known information about a customer."""
        memories = self.memory.get_all(
            filters={"user_id": user_id},
            page=1,
            page_size=200
        )
        return {
            "total_interactions": memories["count"],
            "recent_memories": [
                m["memory"] for m in memories["results"][:10]
            ]
        }

print("✓ SupportBot class defined")
print("  Features: per-user history, ticket tracking, customer context aggregation")

In [ ]:
# [RECIPE] Feedback-Driven Memory Self-Correction

class SelfCorrectingMemory:
    """
    Memory that updates itself when users correct information.
    
    Pattern:
      1. User says something that contradicts a stored memory
      2. Search for the related memory
      3. Update it with the corrected information
      4. Also add the correction as a new fact for audit trail
    """
    
    def __init__(self):
        self.memory = Memory()
    
    def handle_correction(self, user_id: str, correction_message: str):
        """
        Handle a user correction by updating the relevant memory.
        """
        # Search for the fact being corrected
        related = self.memory.search(
            correction_message,
            filters={"user_id": user_id},
            top_k=3
        )
        
        if related["results"]:
            # Update the most relevant memory
            old_memory = related["results"][0]
            self.memory.update(
                memory_id=old_memory["id"],
                data=f"[CORRECTED] {correction_message}",
                metadata={
                    "corrected": True,
                    "previous_value": old_memory["memory"]
                }
            )
            print(f"Updated memory: '{old_memory['memory']}' → '{correction_message}'")
        
        # Also add the correction as a new standalone fact
        self.memory.add(
            messages=[{"role": "user", "content": correction_message}],
            user_id=user_id,
            metadata={"type": "correction"}
        )
        print(f"Added correction as new memory: '{correction_message}'")

# Usage:
# scm = SelfCorrectingMemory()
# scm.handle_correction("alex", "I actually work at Google now, not Netflix.")

print("✓ SelfCorrectingMemory class defined")
print("  Pattern: Find conflicting memory → Update it → Add correction as new fact")

In [ ]:
# [RECIPE] RAG + Mem0 Combined Pipeline

class Mem0RAGPipeline:
    """
    Combine document retrieval (RAG) with user memory (Mem0).
    
    RAG fetches facts from documents (stateless).
    Mem0 provides user context and personalization (stateful).
    Together: personalized, factually-grounded responses.
    """
    
    def __init__(self, vector_db, openai_client=None):
        """
        Args:
            vector_db: Your document vector store (Pinecone, Chroma, etc.)
            openai_client: Optional pre-configured OpenAI client
        """
        self.vector_db = vector_db      # Document store for RAG
        self.memory = Memory()          # User memory store
        self.llm = openai_client or OpenAI()
    
    def query(self, question: str, user_id: str) -> str:
        """
        Answer a question using both documents and user memory.
        """
        # 1. Retrieve relevant documents (RAG)
        docs = self.vector_db.similarity_search(question, k=3)
        doc_context = "\n".join(
            f"[Doc] {d.page_content}" for d in docs
        )
        
        # 2. Retrieve user memories (Mem0)
        memories = self.memory.search(
            question,
            filters={"user_id": user_id},
            top_k=5
        )
        memory_context = "\n".join(
            f"[Memory] {m['memory']}" for m in memories["results"]
        )
        
        # 3. Combined prompt
        response = self.llm.chat.completions.create(
            model="gpt-4.1-mini",
            messages=[{
                "role": "system",
                "content": f"""You are a helpful assistant.

RELEVANT DOCUMENTS (use these for factual grounding):
{doc_context if doc_context else 'No relevant documents found.'}

USER CONTEXT (from past interactions — use for personalization):
{memory_context if memory_context else 'No prior user context.'}

Answer questions using BOTH documents (for facts) and user context (for personalization).
If documents and context conflict, prioritize documents for facts."""
            }, {
                "role": "user",
                "content": question
            }]
        )
        answer = response.choices[0].message.content
        
        # 4. Store this interaction
        self.memory.add([
            {"role": "user", "content": question},
            {"role": "assistant", "content": answer}
        ], user_id=user_id, metadata={"pipeline": "rag+mem0"})
        
        return answer

print("✓ Mem0RAGPipeline class defined")
print("  Combines: RAG (document facts) + Mem0 (user personalization)")
print("  Use case: Personalized, factually-grounded AI responses")

In [ ]:
# [RECIPE] Observability Wrapper — Log All Memory Operations

import logging
from datetime import datetime

class ObservableMemory:
    """
    Wraps a Mem0 Memory instance and logs all operations.
    Useful for debugging, monitoring, and cost tracking.
    """
    
    def __init__(self, memory: Memory, logger: logging.Logger = None):
        self.memory = memory
        self.logger = logger or logging.getLogger("mem0.observable")
        if not self.logger.handlers:
            handler = logging.StreamHandler()
            handler.setFormatter(
                logging.Formatter("%(asctime)s [%(levelname)s] %(message)s")
            )
            self.logger.addHandler(handler)
            self.logger.setLevel(logging.INFO)
    
    def add(self, messages, **kwargs):
        start = datetime.now()
        result = self.memory.add(messages, **kwargs)
        elapsed = (datetime.now() - start).total_seconds()
        self.logger.info(
            f"ADD | user={kwargs.get('user_id', '?')} | "
            f"msgs={len(messages)} | facts={len(result)} | "
            f"{elapsed:.2f}s"
        )
        return result
    
    def search(self, query, **kwargs):
        start = datetime.now()
        result = self.memory.search(query, **kwargs)
        elapsed = (datetime.now() - start).total_seconds()
        self.logger.info(
            f"SEARCH | q='{query[:60]}' | "
            f"results={len(result.get('results', []))} | "
            f"rerank={kwargs.get('rerank', False)} | {elapsed:.2f}s"
        )
        return result
    
    def get_all(self, **kwargs):
        start = datetime.now()
        result = self.memory.get_all(**kwargs)
        elapsed = (datetime.now() - start).total_seconds()
        self.logger.info(
            f"GET_ALL | count={result.get('count', 0)} | {elapsed:.2f}s"
        )
        return result
    
    def update(self, memory_id, data, **kwargs):
        start = datetime.now()
        result = self.memory.update(memory_id=memory_id, data=data, **kwargs)
        elapsed = (datetime.now() - start).total_seconds()
        self.logger.info(
            f"UPDATE | id={memory_id[:12]}... | {elapsed:.2f}s"
        )
        return result
    
    def delete(self, memory_id):
        start = datetime.now()
        result = self.memory.delete(memory_id=memory_id)
        elapsed = (datetime.now() - start).total_seconds()
        self.logger.info(
            f"DELETE | id={memory_id[:12]}... | {elapsed:.2f}s"
        )
        return result

# Usage:
# obs_memory = ObservableMemory(Memory())
# obs_memory.add([{"role": "user", "content": "test"}], user_id="alex")

print("✓ ObservableMemory class defined")
print("  Logs: timing, operation type, result counts for all memory ops")
print("  Use for: debugging, monitoring, cost tracking")

In [ ]:
# [RECIPE] Session-Based Learning Agent

import uuid

class SessionLearningAgent:
    """
    Agent with session-scoped working memory + long-term cross-session memory.
    
    Uses Mem0's run_id to:
      - Keep session-specific context (deleted at session end)
      - Maintain long-term knowledge across sessions (persisted)
    """
    
    def __init__(self):
        self.memory = Memory()
        self.llm = OpenAI()
    
    def start_session(self, user_id: str) -> str:
        """Begin a new session. Returns the run_id."""
        run_id = str(uuid.uuid4())
        print(f"Session started: {run_id[:8]}...")
        return run_id
    
    def chat(self, user_id: str, run_id: str, message: str) -> str:
        """Chat with combined session + long-term memory."""
        # Long-term memories (across all sessions)
        long_term = self.memory.search(
            message,
            filters={
                "AND": [
                    {"user_id": user_id},
                    {"NOT": [{"run_id": run_id}]}  # exclude current session
                ]
            },
            top_k=3
        )
        
        # Session-specific memories (current session only)
        session = self.memory.search(
            message,
            filters={
                "AND": [
                    {"user_id": user_id},
                    {"run_id": run_id}
                ]
            },
            top_k=3
        )
        
        # Build context
        lt_context = "\n".join(m["memory"] for m in long_term["results"])
        s_context = "\n".join(m["memory"] for m in session["results"])
        
        # ... LLM call with both contexts ...
        
        # Store in current session
        self.memory.add(
            [{"role": "user", "content": message}],
            user_id=user_id,
            run_id=run_id
        )
        
        return "response"  # placeholder
    
    def end_session(self, user_id: str, run_id: str):
        """Clean up session-scoped memories (keep long-term)."""
        self.memory.delete_all(user_id=user_id, run_id=run_id)
        print(f"Session {run_id[:8]}... cleaned up")

print("✓ SessionLearningAgent class defined")
print("  Separates: working memory (session) vs long-term memory (all sessions)")
print("  Cleanup: delete session memories at end, keep long-term facts")

## 15. Best Practices Summary

Key takeaways from this tutorial.

In [ ]:
print("=" * 60)
print("MEM0 BEST PRACTICES")
print("=" * 60)
print()
print("🔧 Fact Extraction:")
print("  1. Keep LLM temperature ≤ 0.2 for deterministic extraction")
print("  2. Use custom_instructions to scope extraction to your domain")
print("  3. Include negative examples in few-shot prompts")
print("  4. Spot-check stored memories weekly for quality")
print()
print("🔍 Search & Retrieval:")
print("  1. Always scope searches with entity IDs")
print("  2. Use threshold 0.1-0.3 (raise for precision, lower for recall)")
print("  3. Set top_k = 10-20 for production")
print("  4. Use reference_date for time-sensitive queries")
print("  5. A/B test reranked vs non-reranked before rollout")
print()
print("⚡ Performance:")
print("  1. Use Platform's batch_update/batch_delete for bulk ops")
print("  2. Use AsyncMemory for concurrent workloads")
print("  3. Keep images under 5 MB for multimodal")
print("  4. Minimize reranker top_k for speed")
print()
print("🧹 Data Hygiene:")
print("  1. Implement TTL/deletion for session memories")
print("  2. Version your custom_instructions")
print("  3. Monitor memory growth — prune stale facts")
print("  4. Separate concerns with user_id/agent_id/run_id scopes")
print()
print("🔒 Security:")
print("  1. Never expose API keys in code")
print("  2. Scope Platform API keys to specific projects")
print("  3. Use environment variables for all secrets")
print("  4. Audit delete_all calls (wildcard '*' is powerful!)")
print()
print("=" * 60)
print("Happy building with Mem0! 🧠")
print("=" * 60)

---

## References

- **GitHub:** <https://github.com/mem0ai/mem0>
- **Documentation:** <https://docs.mem0.ai>
- **Platform Dashboard:** <https://app.mem0.ai>
- **PyPI:** <https://pypi.org/project/mem0ai/>
- **npm:** <https://www.npmjs.com/package/mem0ai>
- **Discord:** <https://mem0.dev/DiD>

*Notebook created 2026-06-22. Mem0 v2.0.0.*